# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pathakadithi/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Method choice

I chose **Logistic Regression** because my lane is Refresh / Content Opportunity Scoring, where the goal is to predict whether a page is a refresh opportunity.

Logistic Regression is a good next step after the Week-4 rule-based baseline because it is simple, fast, and interpretable. It can learn how the existing features such as impressions, clicks, CTR, and position relate to the refresh-opportunity target without adding unnecessary model complexity.

I will compare the Logistic Regression model with my Week-4 baseline using the same evaluation data and metric, so the comparison is fair.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Split design

I use the **same train/test split as my Week-4 baseline** so that the model and baseline are evaluated on the same data.

I keep the evaluation set separate from training so the Logistic Regression model does not learn from the rows used to measure its performance. This makes the model-versus-baseline comparison fair and avoids changing the evaluation setup just to improve the model result.

This split is honest for the question because I want to know whether the learned model can improve on the existing baseline on the same content-opportunity prediction task.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [2]:
from datasets import load_dataset

dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_query_90d"
)

df = dataset["train"].to_pandas()

print("Rows:", len(df))
print("Columns:", df.columns.tolist())

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

fact_content_query_90d.parquet: reconstructing file:   0%|          |  0.00B / 60.7MB            

fact_content_query_90d.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2414248 [00:00<?, ? examples/s]

Rows: 2414248
Columns: ['client_hash_id', 'content_hash_id', 'query_hash_id', 'query_char_count', 'query_token_count', 'window_start', 'window_end', 'impressions_90d', 'clicks_90d', 'impressions_last30', 'clicks_last30', 'impressions_prev30', 'clicks_prev30', 'avg_position_90d', 'avg_position_last30', 'avg_position_prev30', 'content_total_impressions_90d', 'content_visible_query_count', 'rare_query_count', 'rare_impressions_share', 'anonymized_impressions_share']


In [3]:
print(df[[
    "impressions_90d",
    "clicks_90d",
    "impressions_last30",
    "clicks_last30",
    "impressions_prev30",
    "clicks_prev30",
    "avg_position_90d"
]].head())

   impressions_90d  clicks_90d  impressions_last30  clicks_last30  \
0               11           0                   0              0   
1               13           0                   0              0   
2               16           0                  11              0   
3               55           0                   1              0   
4               14           0                   0              0   

   impressions_prev30  clicks_prev30  avg_position_90d  
0                  11              0         10.818182  
1                   1              0          1.769231  
2                   5              0         23.562500  
3                   1              0          2.200000  
4                   0              0          3.428571  


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Create CTR features
# ------------------------------------------------------------

df["ctr_90d"] = np.where(
    df["impressions_90d"] > 0,
    df["clicks_90d"] / df["impressions_90d"],
    0
)

df["ctr_last30"] = np.where(
    df["impressions_last30"] > 0,
    df["clicks_last30"] / df["impressions_last30"],
    0
)

df["ctr_prev30"] = np.where(
    df["impressions_prev30"] > 0,
    df["clicks_prev30"] / df["impressions_prev30"],
    0
)

# ------------------------------------------------------------
# 2. Recreate W04 baseline
# ------------------------------------------------------------

df["position_bucket"] = pd.cut(
    df["avg_position_90d"],
    bins=[0, 3, 10, 20, np.inf],
    labels=["1-3", "4-10", "11-20", "21+"],
    include_lowest=True
)

position_ctr = (
    df.groupby("position_bucket", observed=False)["ctr_90d"]
    .mean()
)

df["expected_ctr"] = (
    df["position_bucket"]
    .map(position_ctr)
    .astype(float)
)

df["ctr_gap"] = (
    df["expected_ctr"] - df["ctr_90d"]
).fillna(0)

df["recent_ctr_decline"] = (
    df["ctr_prev30"] - df["ctr_last30"]
).fillna(0)

ctr_gap_scale = df["ctr_gap"].quantile(0.95)
decline_scale = df["recent_ctr_decline"].quantile(0.95)

df["ctr_gap_score"] = (
    df["ctr_gap"] / ctr_gap_scale
).clip(0, 1)

df["decline_score"] = (
    df["recent_ctr_decline"] / decline_scale
).clip(0, 1)

df["baseline_score"] = (
    0.5 * df["ctr_gap_score"]
    + 0.5 * df["decline_score"]
)

print(df["baseline_score"].describe())

count    153844.000000
mean          0.284524
std           0.254010
min           0.000000
25%           0.000000
50%           0.500000
75%           0.500000
max           0.999233
Name: baseline_score, dtype: float64


In [5]:
# Create a binary target from the W04 baseline action
df["baseline_action"] = (
    df["baseline_score"] > 0
).astype(int)

print("Target distribution:")
print(df["baseline_action"].value_counts())
print("\nTarget percentage:")
print(df["baseline_action"].value_counts(normalize=True))

Target distribution:
baseline_action
0    2323545
1      90703
Name: count, dtype: int64

Target percentage:
baseline_action
0    0.96243
1    0.03757
Name: proportion, dtype: float64


In [6]:
from sklearn.model_selection import train_test_split

features = [
    "impressions_90d",
    "clicks_90d",
    "ctr_90d",
    "avg_position_90d",
    "avg_position_last30"
]

X = df[features]
y = df["baseline_action"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True))

Training rows: 1931398
Test rows: 482850

Training target distribution:
baseline_action
0    0.96243
1    0.03757
Name: proportion, dtype: float64

Test target distribution:
baseline_action
0    0.962429
1    0.037571
Name: proportion, dtype: float64


In [8]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("logistic_regression", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))

Accuracy : 0.9528031479755618
Precision: 0.44319515056707076
Recall   : 0.9995038862245742
F1 Score : 0.6140924254483261


In [9]:
from sklearn.metrics import confusion_matrix

# W04 baseline predictions on the same test rows
baseline_pred = df.loc[X_test.index, "baseline_action"]

# Compare Logistic Regression with W04 baseline
comparison = pd.DataFrame({
    "baseline": baseline_pred,
    "logistic_regression": y_pred
}, index=X_test.index)

print("Baseline positive rate:",
      comparison["baseline"].mean())

print("Logistic Regression positive rate:",
      comparison["logistic_regression"].mean())

print("\nAgreement between baseline and model:",
      (comparison["baseline"] == comparison["logistic_regression"]).mean())

print("\nConfusion matrix:")
print(confusion_matrix(
    comparison["baseline"],
    comparison["logistic_regression"]
))

Baseline positive rate: 0.037570674122398263
Logistic Regression positive rate: 0.08473024748886818

Agreement between baseline and model: 0.9528031479755618

Confusion matrix:
[[441929  22780]
 [     9  18132]]


Logistic Regression was trained using the same 80/20 stratified split and the five Lane 2 features. The model achieved 95.28% accuracy, 44.32% precision, 99.95% recall, and 0.614 F1 against the W04 baseline labels. It identified 8.47% of test rows as opportunities compared with 3.76% from the baseline. The model had only 9 false negatives, so it captured nearly all opportunities identified by the baseline, but it also produced additional candidates. Therefore, the model provides a broader opportunity set rather than simply reproducing the baseline.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Error analysis

comparison["error_type"] = "correct"
comparison.loc[
    (comparison["baseline"] == 0) &
    (comparison["logistic_regression"] == 1),
    "error_type"
] = "false_positive"

comparison.loc[
    (comparison["baseline"] == 1) &
    (comparison["logistic_regression"] == 0),
    "error_type"
] = "false_negative"

print("Error counts:")
print(comparison["error_type"].value_counts())

# Logistic Regression coefficients
coefficients = pd.Series(
    model.named_steps["logistic_regression"].coef_[0],
    index=features
).sort_values(key=abs, ascending=False)

print("\nFeature coefficients:")
print(coefficients)

# Compare feature averages for false positives and true positives
fp_idx = comparison.index[comparison["error_type"] == "false_positive"]
tp_idx = comparison.index[comparison["error_type"] == "correct"]

print("\nFalse-positive feature means:")
print(df.loc[fp_idx, features].mean())

print("\nTrue-positive feature means:")
print(df.loc[
    comparison.index[
        (comparison["baseline"] == 1) &
        (comparison["logistic_regression"] == 1)
    ],
    features
].mean())


Error counts:
error_type
correct           460061
false_positive     22780
false_negative         9
Name: count, dtype: int64

Feature coefficients:
clicks_90d             6.805755
impressions_90d        0.719535
avg_position_90d      -0.502903
ctr_90d                0.303209
avg_position_last30   -0.065149
dtype: float64

False-positive feature means:
impressions_90d        330.217340
clicks_90d               1.858780
ctr_90d                  0.026401
avg_position_90d         7.259388
avg_position_last30      8.847485
dtype: float64

True-positive feature means:
impressions_90d        643.314582
clicks_90d               2.790646
ctr_90d                  0.021435
avg_position_90d         6.820698
avg_position_last30      8.381268
dtype: float64


The model's main error is false positives: it predicted 22,780 rows as opportunities that were not marked as opportunities by the W04 baseline. It made only 9 false-negative errors, meaning it captured almost all baseline opportunities.

The model leans most strongly on clicks_90d, followed by impressions_90d. The positive coefficient for clicks means higher clicks strongly influence the model toward the opportunity class, while avg_position_90d has a negative coefficient, so better ranking positions tend to reduce the predicted opportunity probability.

The false positives have lower average impressions and clicks than the true positives, but their average CTR is slightly higher. This suggests the model is sensitive to traffic and click volume and may identify some additional pages that the W04 rule-based baseline does not consider opportunities.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.